# GPT · Jev 검색 방식 분류 비교
팀원의 Search Worker 프롬프트·SearchMode·모델 설정과 Jev Choice 호출 방식을 재사용하는 독립 실험입니다. 기존 create_agent의 도구 실행을 뺀 **분류 전용 비교**이며, Worker 전체의 속도 비교가 아닙니다. Aura·임베딩·답변 생성은 실행하지 않습니다.

프로젝트 .venv 커널에서 위부터 실행하세요. API 실행은 기본 OFF입니다. config.py의 충돌 표시는 수정하지 않고 설정만 읽습니다.

## 1. 프로젝트 및 팀원 코드
프롬프트와 출력 타입을 직접 불러옵니다. config 전체 import는 하지 않습니다.

In [ ]:
from pathlib import Path
import asyncio, hashlib, importlib.util, json, os, random, re, sys, time
from datetime import datetime, timezone
from typing import get_args
import pandas as pd
from IPython.display import display
from dotenv import dotenv_values
from pydantic import BaseModel

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "app/core/prompts.py").exists()),
    Path(r"C:\Users\Playdata\Desktop\mle-01-p3-team3"))
assert (PROJECT_ROOT / "app/core/prompts.py").exists(), "PROJECT_ROOT를 확인하세요."
def load_local(name, path):
    spec = importlib.util.spec_from_file_location(name, PROJECT_ROOT / path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module
prompts = load_local("_compare_prompts", "app/core/prompts.py")
schemas = load_local("_compare_schemas", "app/core/schemas.py")
SearchMode = schemas.SearchMode
MODES = tuple(get_args(SearchMode))
assert set(MODES) == {"vector", "graph", "hybrid"}
config_text = (PROJECT_ROOT / "app/core/config.py").read_text(encoding="utf-8-sig")
HAS_CONFLICT = bool(re.search(r"^(<<<<<<<|=======|>>>>>>>)", config_text, re.M))
print("프로젝트:", PROJECT_ROOT)
print("config.py 충돌 표시:", HAS_CONFLICT, "/ 기존 파일 수정 없이 설정을 읽습니다.")


## 2. 모델과 실행 설정
환경변수 → .env → 팀원 모델 기본값 순서입니다. 기본 6문항×3회×2모델+워밍업 2회=38회 API 호출. 전체 18문항은 110회입니다.

In [ ]:
env_values = dotenv_values(PROJECT_ROOT / ".env")
def setting(key, default=""):
    return os.environ.get(key) or env_values.get(key) or default
def team_default(field):
    matches = re.findall(r'^    ' + re.escape(field) + r': str = ["\x27]([^"\x27]+)["\x27]', config_text, re.M)
    if len(set(matches)) != 1:
        raise ValueError(field + " 모델 기본값 확인 필요")
    return matches[0]
GPT_MODEL = setting("OPENAI_CHAT_MODEL", team_default("openai_chat_model"))
JEV_MODEL = setting("TYPESAFE_MODEL", team_default("typesafe_model"))
match = re.search(r'reasoning_effort=["\x27]([^"\x27]+)["\x27]', config_text)
assert match, "팀원 reasoning 설정 확인 필요"
GPT_REASONING = match[1]
RUN_API = False
GOLD_REVIEWED = False
CASE_LIMIT = 6  # 전체 18문항은 None
REPEATS = 3
TIMEOUT_SECONDS = 40
SEED = 20261113
print({"GPT": GPT_MODEL, "Jev": JEV_MODEL, "reasoning_effort": GPT_REASONING})


## 3. 공통 정책
팀원 프롬프트 뒤에 분류 전용 출력 지시를 동일하게 붙입니다. 정확도는 이 정책과의 일치율이며 유일한 검색 전략이라는 뜻은 아닙니다.

In [ ]:
CRITERIA = {
    "vector": "업무 내용으로 직업을 탐색하거나 특정 직업의 업무 설명만 요청한다.",
    "graph": "직업·학과·능력·자격이 명시되어 있고 관계 또는 경로만 요청한다. 앞 대화에 대상이 있어도 포함한다.",
    "hybrid": "업무 설명으로 직업을 찾고 그 후보의 관계를 요청하거나 업무 설명과 관계 정보를 함께 요청한다.",
}
DECISION_PROMPT = prompts.SEARCH_SYSTEM_PROMPT + """
[독립 분류 실험의 출력 범위]
도구 실행·검색·근거 생성·Cypher 작성은 하지 않는다.
위 검색 결과 반환 지시 대신 이번 출력 스키마에 따라 검색 방식 하나만 반환한다.
질문과 최근 대화를 함께 참고한다. 공통 선택 정책:
""" + json.dumps(CRITERIA, ensure_ascii=False)
class RouteResult(BaseModel):
    search_mode: SearchMode
def payload(case):
    return {"question": case["question"], "recent_conversation": case["history"],
            "previous_search_mode": None, "verification_feedback": None}


## 4. 정답 초안 18개
유형별 6개이며 팀 검토 후 GOLD_REVIEWED=True로 설정하세요. 인사·공격·대상 불명확 질문은 별도 평가 대상이며 여기서는 제외합니다.

In [ ]:
triples = [
    ("날씨 데이터를 분석하고 예측하는 직업을 찾아줘", "일기예보관과 관련된 학과는?", "날씨 데이터를 분석하는 직업과 관련 학과를 알려줘"),
    ("음식을 만들고 메뉴를 개발하는 직업을 찾아줘", "조리사와 관련된 자격은?", "음식을 만들고 메뉴를 개발하는 직업과 관련 자격을 알려줘"),
    ("자료에서 패턴을 분석하는 직업을 찾아줘", "소방관과 연결된 핵심능력을 알려줘", "자료에서 패턴을 분석하는 직업과 핵심능력을 알려줘"),
    ("일기예보관은 무슨 일을 해?", "같은 능력 노드에 연결된 직업들의 경로를 보여줘", "일기예보관의 업무 설명과 관련 학과를 알려줘"),
    ("뉴스를 전달하는 일을 하는 직업을 찾아줘", "일기예보관의 관련 학과를 거쳐 연결되는 다른 직업은?", "뉴스를 전달하는 직업과 관련 학과를 알려줘"),
    ("그 직업의 업무 내용을 더 설명해줘", "그 직업의 관련 학과는?", "그 직업의 업무 내용과 관련 학과를 함께 알려줘"),
]
CASES = []
for i, questions in enumerate(triples):
    for mode, question in zip(("vector", "graph", "hybrid"), questions):
        CASES.append({"case_id": f"{mode[0]}{i+1:02}", "question": question, "expected": mode,
            "history": [{"role": "user", "content": "일기예보관에 대해 알아보고 있어."}] if i == 5 else []})
assert len(CASES) == 18 and len({c["case_id"] for c in CASES}) == 18
display(pd.DataFrame(CASES))


## 5. 모델별 호출
GPT는 팀원 모델·reasoning 설정으로 구조화 분류를 수행합니다. Jev는 router.py의 Choice → choices → choice 패턴을 재사용합니다. GPT 재시도는 0이며 Jev SDK 내부 동작과 같다고 가정하지 않습니다. 오류 전문과 키는 출력하지 않습니다.

In [ ]:
def build_classifiers():
    from langchain_openai import ChatOpenAI
    from langchain_typesafe import Choice, TypeSafeClassifier
    gpt_key, jev_key = setting("OPENAI_API_KEY"), setting("TYPESAFE_API_KEY")
    if not gpt_key or not jev_key:
        raise ValueError("프로젝트 .env의 API 키 두 개를 확인하세요.")
    gpt = ChatOpenAI(model=GPT_MODEL, api_key=gpt_key, reasoning_effort=GPT_REASONING,
        timeout=TIMEOUT_SECONDS, max_retries=0).with_structured_output(RouteResult, method="json_schema")
    jev = TypeSafeClassifier(model=JEV_MODEL, api_key=jev_key, timeout=TIMEOUT_SECONDS)
    async def gpt_route(state):
        result = await gpt.ainvoke([("system", DECISION_PROMPT),
                                    ("human", json.dumps(state, ensure_ascii=False))])
        return result.search_mode
    async def jev_route(state):
        decision = await jev.ainvoke({"state": state,
            "questions": {"route": Choice(instructions=DECISION_PROMPT, criteria=CRITERIA)}})
        answer = decision.choices.get("route")
        if answer is None or answer.choice not in MODES:
            raise ValueError("유효한 Jev 분류 없음")
        return answer.choice
    return {"GPT": gpt_route, "Jev": jev_route}, jev

async def measure(fn, state):
    start = time.perf_counter()
    try:
        predicted = await asyncio.wait_for(fn(state), timeout=TIMEOUT_SECONDS)
        if predicted not in MODES:
            raise ValueError("분류 범위 밖 출력")
        return {"predicted": predicted, "error_type": None, "latency_ms": (time.perf_counter()-start)*1000}
    except Exception as exc:
        return {"predicted": None, "error_type": type(exc).__name__, "latency_ms": (time.perf_counter()-start)*1000}


## 6. 비교 실행
문항 순서는 섞고 모델 호출 순서는 번갈아 배치합니다. 같은 설정으로 다시 실행하면 완료된 측정은 건너뜁니다. 오류 기록도 보존합니다. 워밍업은 재실행 시 다시 호출하며 집계에서 제외합니다.

In [ ]:
ACTIVE_CASES = CASES if CASE_LIMIT is None else CASES[:CASE_LIMIT]
assert ACTIVE_CASES and REPEATS > 0
fingerprint = hashlib.sha256(json.dumps({"prompt": DECISION_PROMPT, "cases": ACTIVE_CASES,
    "gpt": GPT_MODEL, "jev": JEV_MODEL, "reasoning": GPT_REASONING, "repeats": REPEATS,
    "seed": SEED, "timeout": TIMEOUT_SECONDS}, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
if globals().get("experiment_id") != fingerprint:
    records = []
    experiment_id = fingerprint

if not RUN_API:
    print("API 호출 OFF. 셀 2에서 RUN_API / GOLD_REVIEWED 설정을 변경하세요.")
else:
    if not GOLD_REVIEWED:
        raise ValueError("정답 초안을 검토하고 GOLD_REVIEWED=True로 설정하세요.")
    classifiers, jev_client = build_classifiers()
    try:
        warmups = []
        for name, fn in classifiers.items():
            warmups.append({"provider": name, **await measure(fn, payload(ACTIVE_CASES[0]))})
        display(pd.DataFrame(warmups))
        if any(r["error_type"] for r in warmups):
            raise RuntimeError("워밍업 실패. 본 측정 미실행. 모델명·키·네트워크 확인 필요.")
        done = {(r["repeat"], r["case_id"], r["provider"]) for r in records}
        rng = random.Random(SEED)
        for repeat in range(1, REPEATS+1):
            ordered = list(ACTIVE_CASES)
            rng.shuffle(ordered)
            for position, case in enumerate(ordered):
                names = ["GPT", "Jev"] if (repeat+position)%2 else ["Jev", "GPT"]
                for order, name in enumerate(names):
                    key = (repeat, case["case_id"], name)
                    if key in done:
                        continue
                    result = await measure(classifiers[name], payload(case))
                    records.append({"experiment_id": experiment_id, "repeat": repeat,
                        "case_id": case["case_id"], "question": case["question"], "expected": case["expected"],
                        "provider": name, "model": GPT_MODEL if name=="GPT" else JEV_MODEL,
                        "pair_order": order, "timestamp": datetime.now(timezone.utc).isoformat(), **result,
                        "correct": result["error_type"] is None and result["predicted"]==case["expected"]})
                    done.add(key)
                print(f"반복 {repeat}/{REPEATS}: {case['case_id']} 완료")
    finally:
        if getattr(jev_client, "async_client", None) is not None:
            await jev_client.async_client.aclose()
        if getattr(jev_client, "client", None) is not None:
            jev_client.client.close()
    print("저장된 측정:", len(records))


## 7. 결과 비교
오류도 전체 정확도에서 오답으로 포함합니다. 성공 호출 지연시간과 오류를 포함한 시간을 구분합니다. 반복은 독립된 질문 수가 아니며 작은 표본의 p95를 일반화하지 마세요.

In [ ]:
def summarize(rows):
    frame = pd.DataFrame(rows)
    summary = []
    for name, group in frame.groupby("provider"):
        valid = group[group["error_type"].isna()]
        summary.append({"provider": name, "calls": len(group), "accuracy_all": group["correct"].mean(),
            "error_rate": group["error_type"].notna().mean(), "median_all_ms": group["latency_ms"].median(),
            "median_success_ms": valid["latency_ms"].median(), "p95_success_ms": valid["latency_ms"].quantile(.95)})
    return frame, pd.DataFrame(summary)
if records:
    results, summary = summarize(records)
    display(summary.round(3))
    display(results.groupby(["provider", "expected"])["correct"].agg(["count", "mean"]))
    display(pd.crosstab([results["provider"], results["expected"]], results["predicted"].fillna("ERROR")))
    display(results[~results["correct"]][["provider", "case_id", "expected", "predicted", "error_type", "latency_ms"]])
    changes = results.assign(choice=results["predicted"].fillna("ERROR")).groupby(["provider", "case_id"])["choice"].nunique()
    print("반복 중 분류가 달라진 문항 수:", changes.gt(1).groupby(level=0).sum().to_dict())
    print("완료/예정:", len(results), "/", len(ACTIVE_CASES)*REPEATS*2)
else:
    print("측정 결과 없음. API 실행 전입니다.")


## 8. 선택적 CSV 저장
기본 OFF. True일 때만 새 output 폴더에 CSV와 재현 정보를 생성합니다. 키나 환경변수 전체는 저장하지 않습니다.

In [ ]:
EXPORT_RESULTS = False
if EXPORT_RESULTS and records:
    folder = PROJECT_ROOT / "output" / "routing_compare" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    folder.mkdir(parents=True, exist_ok=False)
    frame, summary = summarize(records)
    frame.to_csv(folder / "trials.csv", index=False, encoding="utf-8-sig")
    summary.to_csv(folder / "summary.csv", index=False, encoding="utf-8-sig")
    metadata = {"experiment_id": experiment_id, "cases": ACTIVE_CASES, "repeats": REPEATS,
        "seed": SEED, "gpt_model": GPT_MODEL, "jev_model": JEV_MODEL, "reasoning_effort": GPT_REASONING,
        "config_had_conflicts": HAS_CONFLICT, "decision_prompt": DECISION_PROMPT,
        "scope": "classification only; not original full Search Worker",
        "source_sha256": {p: hashlib.sha256((PROJECT_ROOT/p).read_bytes()).hexdigest()
            for p in ["app/core/prompts.py", "app/core/schemas.py", "app/core/config.py", "app/guards/router.py"]}}
    (folder / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    print("결과 저장:", folder)
else:
    print("추가 파일 저장 OFF 또는 측정 결과 없음.")


## 결과 해석
Jev가 분류에서 빨라도 검색 인자·Cypher 생성까지 대체한 결과는 아닙니다. 소규모 수동 정답 초안이므로 별도의 미사용 질문으로 재검증하세요. 정확도·지연시간에서 이점이 있을 때 팀원과 실제 검색 연결 실험으로 진행합니다.